# Phase 3 — Forecasting, Scenarios, and Multi-Scale Analysis

This notebook extends the project into **Phase 3**:
- fits a municipio-level forecasting model
- generates **municipal, regional, and island** forecasts
- compares **baseline (no natural disaster)** against:
  - a **hurricane scenario** (configurable category and year)
  - an **earthquake scenario** (configurable magnitude and year)
- creates **side-by-side forecast fan charts** similar to the example plot

## Important note on scenario interpretation
The scenario engine maps hypothetical future events into the model's engineered disaster features:

- **Hurricane category** → a stylized addition to `wind_3yr_sum`
- **Earthquake magnitude** → a stylized addition to `seismic_3yr_sum`

These are **model-based scenario simulations**, not deterministic physical forecasts. They are useful for:
- policy analysis
- stress testing
- comparing *relative* population trajectories under different hazard regimes


In [1]:

# Core imports
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(42)


In [2]:

# Output folder
OUTDIR = Path("phase3_outputs")
OUTDIR.mkdir(exist_ok=True, parents=True)

saved_files = []

def save_csv(df, name):
    path = OUTDIR / name
    df.to_csv(path, index=False)
    saved_files.append({"file": name, "path": str(path.resolve()), "type": "csv"})
    print(f"Saved: {path.resolve()}")
    return path

def save_json(obj, name):
    path = OUTDIR / name
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    saved_files.append({"file": name, "path": str(path.resolve()), "type": "json"})
    print(f"Saved: {path.resolve()}")
    return path

def save_figure(fig, name, dpi=150):
    path = OUTDIR / name
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    saved_files.append({"file": name, "path": str(path.resolve()), "type": "figure"})
    print(f"Saved: {path.resolve()}")
    return path


In [3]:

# Load source data
candidate_files = [
    Path("processed_puerto_rico_data_rebuild.csv"),
    Path("processed_puerto_rico_data_enriched.csv"),
    Path("/mnt/data/processed_puerto_rico_data_rebuild.csv"),
    Path("/mnt/data/processed_puerto_rico_data_enriched.csv"),
]

data_path = None
for p in candidate_files:
    if p.exists():
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("Could not find processed Puerto Rico Phase 1 dataset.")

df = pd.read_csv(data_path)
print("Loaded:", data_path)
print("Shape:", df.shape)
print("Years:", df["year"].min(), "to", df["year"].max())
print("Municipalities:", df["municipio"].nunique())


Loaded: processed_puerto_rico_data_rebuild.csv
Shape: (1170, 160)
Years: 2010 to 2024
Municipalities: 78


In [4]:

# Region mapping from the project specification
region_map = {
    # Metro
    "San Juan": "Metro",
    "Bayamón": "Metro",
    "Carolina": "Metro",
    "Cataño": "Metro",
    "Guaynabo": "Metro",
    "Toa Alta": "Metro",
    "Toa Baja": "Metro",
    "Trujillo Alto": "Metro",

    # North
    "Arecibo": "North",
    "Barceloneta": "North",
    "Camuy": "North",
    "Dorado": "North",
    "Florida": "North",
    "Hatillo": "North",
    "Manatí": "North",
    "Quebradillas": "North",
    "Vega Alta": "North",
    "Vega Baja": "North",

    # South
    "Arroyo": "South",
    "Coamo": "South",
    "Guayama": "South",
    "Guayanilla": "South",
    "Juana Díaz": "South",
    "Patillas": "South",
    "Peñuelas": "South",
    "Ponce": "South",
    "Salinas": "South",
    "Santa Isabel": "South",
    "Villalba": "South",
    "Yauco": "South",

    # West
    "Aguada": "West",
    "Aguadilla": "West",
    "Añasco": "West",
    "Cabo Rojo": "West",
    "Guánica": "West",
    "Hormigueros": "West",
    "Isabela": "West",
    "Lajas": "West",
    "Las Marías": "West",
    "Maricao": "West",
    "Mayagüez": "West",
    "Moca": "West",
    "Rincón": "West",
    "Sabana Grande": "West",
    "San Germán": "West",
    "San Sebastián": "West",

    # East
    "Canóvanas": "East",
    "Ceiba": "East",
    "Fajardo": "East",
    "Humacao": "East",
    "Juncos": "East",
    "Las Piedras": "East",
    "Loíza": "East",
    "Luquillo": "East",
    "Maunabo": "East",
    "Naguabo": "East",
    "Río Grande": "East",
    "San Lorenzo": "East",
    "Yabucoa": "East",
    "Caguas": "East",
    "Gurabo": "East",
    "Culebra": "East",
    "Vieques": "East",

    # Central Mountains
    "Adjuntas": "Central Mountains",
    "Aguas Buenas": "Central Mountains",
    "Aibonito": "Central Mountains",
    "Barranquitas": "Central Mountains",
    "Cayey": "Central Mountains",
    "Ciales": "Central Mountains",
    "Cidra": "Central Mountains",
    "Comerío": "Central Mountains",
    "Corozal": "Central Mountains",
    "Jayuya": "Central Mountains",
    "Lares": "Central Mountains",
    "Morovis": "Central Mountains",
    "Naranjito": "Central Mountains",
    "Orocovis": "Central Mountains",
    "Utuado": "Central Mountains",
}

df["region"] = df["municipio"].map(region_map).fillna("Unmapped")
print(df["region"].value_counts(dropna=False))


region
East                 255
West                 240
Central Mountains    225
South                180
North                150
Metro                120
Name: count, dtype: int64


## Prepare vulnerability and modeling features

The notebook prefers the PCA-based vulnerability signal (`svi_pca_1`).  
If it is missing, it is reconstructed from the 15 SVI indicator columns.


In [5]:

# Rebuild PCA-based SVI if needed
svi_indicator_columns = [
    "poverty_rate_pct",
    "unemployment_rate_pct",
    "per_capita_income_inv",
    "no_hs_diploma_pct",
    "under_18_pct",
    "over_65_pct",
    "disability_pct",
    "single_parent_pct",
    "minority_pct",
    "limited_english_pct",
    "multi_unit_housing_pct",
    "mobile_homes_pct",
    "crowding_pct",
    "no_vehicle_pct",
    "pct_group_quarters",
]

for c in svi_indicator_columns:
    if c not in df.columns:
        raise ValueError(f"Missing required SVI indicator column: {c}")

if "svi_pca_1" not in df.columns:
    svi_tmp = df[svi_indicator_columns].apply(pd.to_numeric, errors="coerce")
    svi_imputed = pd.DataFrame(
        SimpleImputer(strategy="median").fit_transform(svi_tmp),
        columns=svi_indicator_columns,
        index=df.index,
    )
    svi_scaled = StandardScaler().fit_transform(svi_imputed)
    pca = PCA(n_components=1, random_state=42)
    df["svi_pca_1"] = pca.fit_transform(svi_scaled)[:, 0]
    print("Constructed svi_pca_1 from 15 indicators.")
else:
    print("Using existing svi_pca_1 column.")

# Establishment growth fallback
if "establishment_growth" not in df.columns:
    est_col = None
    for c in ["establishments_total_all_sectors", "establishment_count"]:
        if c in df.columns:
            est_col = c
            break
    if est_col is None:
        df["establishment_growth"] = 0.0
    else:
        df[est_col] = pd.to_numeric(df[est_col], errors="coerce")
        df["establishment_growth"] = df.groupby("municipio")[est_col].pct_change() * 100

# Income growth fallback
if "income_growth" not in df.columns:
    if "median_income_real" in df.columns:
        df["median_income_real"] = pd.to_numeric(df["median_income_real"], errors="coerce")
        df["income_growth"] = df.groupby("municipio")["median_income_real"].pct_change() * 100
    else:
        df["income_growth"] = 0.0

# Crime fallback
if "lag_total_crime_rate" not in df.columns:
    if "total_crime_rate" in df.columns:
        df["lag_total_crime_rate"] = df.groupby("municipio")["total_crime_rate"].shift(1)
    else:
        df["lag_total_crime_rate"] = 0.0

# Interaction features
df["wind_3yr_sum"] = pd.to_numeric(df.get("wind_3yr_sum", 0), errors="coerce").fillna(0)
df["seismic_3yr_sum"] = pd.to_numeric(df.get("seismic_3yr_sum", 0), errors="coerce").fillna(0)
df["wind_3yr_x_svi"] = df["wind_3yr_sum"] * df["svi_pca_1"]
df["seismic_3yr_x_svi"] = df["seismic_3yr_sum"] * df["svi_pca_1"]

# Basic type cleanup
for c in ["lag_pop_change_1", "lag_pop_change_2", "income_growth", "establishment_growth", "lag_total_crime_rate", "target_pop_change_1y", "total_population"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values(["municipio", "year"]).reset_index(drop=True)
print(df[["municipio", "year", "svi_pca_1", "wind_3yr_sum", "seismic_3yr_sum"]].head())


Using existing svi_pca_1 column.
  municipio  year  svi_pca_1  wind_3yr_sum  seismic_3yr_sum
0  Adjuntas  2010   3.762319      0.344098     16570.544670
1  Adjuntas  2011   3.340467      2.986699     23284.161997
2  Adjuntas  2012   2.648887      2.986699     26371.233666
3  Adjuntas  2013   2.748966      3.503061     13536.183480
4  Adjuntas  2014   2.942612      1.438839     36702.051446


In [6]:

# Core Phase 3 annual forecasting target and features
TARGET = "target_pop_change_1y"

numeric_features = [
    "lag_pop_change_1",
    "lag_pop_change_2",
    "svi_pca_1",
    "wind_3yr_sum",
    "seismic_3yr_sum",
    "wind_3yr_x_svi",
    "seismic_3yr_x_svi",
    "income_growth",
    "establishment_growth",
    "lag_total_crime_rate",
    "year",
]

categorical_features = ["municipio", "region"]
all_features = numeric_features + categorical_features

missing_needed = [c for c in all_features + [TARGET, "total_population"] if c not in df.columns]
if missing_needed:
    raise ValueError(f"Missing required columns for Phase 3: {missing_needed}")

model_df = df[all_features + [TARGET, "total_population"]].copy()
model_df = model_df.dropna(subset=[TARGET, "total_population"]).reset_index(drop=True)

print("Modeling rows:", len(model_df))
print("Target year coverage:", df.loc[df[TARGET].notna(), "year"].min(), "to", df.loc[df[TARGET].notna(), "year"].max())


Modeling rows: 1092
Target year coverage: 2010 to 2023


## Train annual forecasting model

Phase 3 uses a municipio-level **Extra Trees** model for annual recursive forecasting.  
This is appropriate here because the model:
- handled nonlinearity well in Phase 2,
- can leverage geography via encoded categorical features,
- works well for scenario stress-testing.


In [7]:

# Time-based split for a quick annual target diagnostic
train_mask = model_df["year"] <= 2020
val_mask = model_df["year"].between(2021, 2022)
test_mask = model_df["year"] >= 2023

X_train = model_df.loc[train_mask, all_features]
y_train = model_df.loc[train_mask, TARGET]

X_val = model_df.loc[val_mask, all_features]
y_val = model_df.loc[val_mask, TARGET]

X_test = model_df.loc[test_mask, all_features]
y_test = model_df.loc[test_mask, TARGET]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

annual_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=500,
        random_state=42,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1
    )),
])

annual_model.fit(X_train, y_train)

val_pred = annual_model.predict(X_val) if len(X_val) else np.array([])
test_pred = annual_model.predict(X_test) if len(X_test) else np.array([])

def safe_metrics(y_true, y_pred, split_name):
    if len(y_true) == 0:
        return {
            "split": split_name,
            "n": 0,
            "r2": None,
            "rmse": None,
            "mae": None
        }
    return {
        "split": split_name,
        "n": int(len(y_true)),
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred))
    }

phase3_metrics = [
    safe_metrics(y_val, val_pred, "validation"),
    safe_metrics(y_test, test_pred, "test"),
]
phase3_metrics_df = pd.DataFrame(phase3_metrics)
phase3_metrics_df


,split,n,r2,rmse,mae
0,validation,156,-1.253503,1.784385,1.307751
1,test,78,-0.150687,1.288236,0.649268


In [8]:

# Fit final annual model on all available annual-target rows
annual_model.fit(model_df[all_features], model_df[TARGET])

all_fitted = annual_model.predict(model_df[all_features])
residuals = model_df[TARGET].values - all_fitted

phase3_model_summary = {
    "phase": "Phase 3",
    "model": "ExtraTreesRegressor",
    "target": TARGET,
    "features": all_features,
    "diagnostics": phase3_metrics,
    "residual_mean": float(np.mean(residuals)),
    "residual_std": float(np.std(residuals)),
    "n_training_rows_final": int(len(model_df))
}

save_json(phase3_model_summary, "phase3_model_summary.json")
phase3_model_summary


Saved: /Users/andreruiz/Library/Mobile Documents/com~apple~CloudDocs/Documents/Capstone/CAPSTONE_DATA_SCIENCE_RIVERARUIZ/updated_analysis/phase3_outputs/phase3_model_summary.json


{'phase': 'Phase 3',
 'model': 'ExtraTreesRegressor',
 'target': 'target_pop_change_1y',
 'features': ['lag_pop_change_1',
  'lag_pop_change_2',
  'svi_pca_1',
  'wind_3yr_sum',
  'seismic_3yr_sum',
  'wind_3yr_x_svi',
  'seismic_3yr_x_svi',
  'income_growth',
  'establishment_growth',
  'lag_total_crime_rate',
  'year',
  'municipio',
  'region'],
 'diagnostics': [{'split': 'validation',
   'n': 156,
   'r2': -1.2535027236504699,
   'rmse': 1.784384852551231,
   'mae': 1.3077511993382496},
  {'split': 'test',
   'n': 78,
   'r2': -0.15068720005283365,
   'rmse': 1.288235528200494,
   'mae': 0.6492682814238611}],
 'residual_mean': -4.636096146786368e-17,
 'residual_std': 1.1208425752947802,
 'n_training_rows_final': 1092}

## Scenario settings

You can change these values and rerun the forecast section.

- `shock_year`: the year when the hypothetical event occurs
- `hurricane_category`: category 1–5
- `earthquake_magnitude`: stylized magnitude
- `forecast_end_year`: final forecast year
- `n_sims`: number of simulation paths for fan charts


In [9]:

# Scenario controls
shock_year = 2026
hurricane_category = 3
earthquake_magnitude = 6.5
forecast_start_year = int(df["year"].max()) + 1
forecast_end_year = 2030
n_sims = 300

# Stylized mapping from hurricane category to wind score
category_to_wind_score = {
    1: 75.0,
    2: 90.0,
    3: 105.0,
    4: 125.0,
    5: 145.0,
}

scenario_definitions = {
    "baseline_no_disaster": {
        "type": "none",
        "shock_year": None,
        "wind_score": 0.0,
        "quake_magnitude": 0.0,
        "label": "No natural disaster"
    },
    f"hurricane_cat{hurricane_category}_{shock_year}": {
        "type": "hurricane",
        "shock_year": shock_year,
        "wind_score": category_to_wind_score.get(hurricane_category, 105.0),
        "quake_magnitude": 0.0,
        "label": f"Hurricane Category {hurricane_category} in {shock_year}"
    },
    f"earthquake_m{str(earthquake_magnitude).replace('.', '')}_{shock_year}": {
        "type": "earthquake",
        "shock_year": shock_year,
        "wind_score": 0.0,
        "quake_magnitude": float(earthquake_magnitude),
        "label": f"Earthquake M{earthquake_magnitude} in {shock_year}"
    }
}

scenario_definitions


{'baseline_no_disaster': {'type': 'none',
  'shock_year': None,
  'wind_score': 0.0,
  'quake_magnitude': 0.0,
  'label': 'No natural disaster'},
 'hurricane_cat3_2026': {'type': 'hurricane',
  'shock_year': 2026,
  'wind_score': 105.0,
  'quake_magnitude': 0.0,
  'label': 'Hurricane Category 3 in 2026'},
 'earthquake_m65_2026': {'type': 'earthquake',
  'shock_year': 2026,
  'wind_score': 0.0,
  'quake_magnitude': 6.5,
  'label': 'Earthquake M6.5 in 2026'}}

In [10]:

# Helper tables for recursive forecasting
latest_year = int(df["year"].max())
latest_rows = (
    df.loc[df["year"] == latest_year]
    .sort_values("municipio")
    .reset_index(drop=True)
    .copy()
)

# Municipio-level long-run averages for exogenous variables
exog_cols = ["income_growth", "establishment_growth", "lag_total_crime_rate", "svi_pca_1"]
exog_by_muni = (
    df.groupby("municipio")[exog_cols]
    .mean()
    .reset_index()
)

latest_rows = latest_rows.merge(exog_by_muni, on="municipio", suffixes=("", "_avg"), how="left")

# Starting rolling disaster memory from the last 2 observed years
history_events = (
    df[["municipio", "year", "wind_3yr_sum", "seismic_3yr_sum"]]
    .copy()
)

historical_island = (
    df.groupby("year", as_index=False)["total_population"]
    .sum()
    .rename(columns={"total_population": "population"})
)
historical_region = (
    df.groupby(["year", "region"], as_index=False)["total_population"]
    .sum()
    .rename(columns={"total_population": "population"})
)


In [11]:

def compute_recent_hazard_memory(muni_hist, current_year, scenario, municipio):
    '''
    Build stylized 3-year rolling hazard exposure.
    We combine recent observed exposure history with hypothetical future scenario events.
    '''
    # Historical contribution from actual observed years still within the trailing 3-year window
    hist_window = muni_hist[(muni_hist["year"] >= current_year - 2) & (muni_hist["year"] <= latest_year)]
    hist_wind = hist_window["wind_3yr_sum"].replace([np.inf, -np.inf], np.nan).fillna(0).tail(1)
    hist_quake = hist_window["seismic_3yr_sum"].replace([np.inf, -np.inf], np.nan).fillna(0).tail(1)

    base_wind = float(hist_wind.iloc[0]) if len(hist_wind) else 0.0
    base_quake = float(hist_quake.iloc[0]) if len(hist_quake) else 0.0

    # Decay baseline historical exposure after the observed period
    years_ahead = max(0, current_year - latest_year)
    decay_factor = max(0.0, 1 - 0.35 * years_ahead)
    wind_total = base_wind * decay_factor
    quake_total = base_quake * decay_factor

    # Add hypothetical event if it falls within the trailing 3-year window
    if scenario["shock_year"] is not None and (current_year - 2) <= scenario["shock_year"] <= current_year:
        if scenario["type"] == "hurricane":
            wind_total += scenario["wind_score"]
        elif scenario["type"] == "earthquake":
            quake_total += scenario["quake_magnitude"]

    return wind_total, quake_total


def build_future_row(prev_row, year, scenario, muni_hist):
    row = prev_row.copy()
    row["year"] = year

    municipio = row["municipio"]
    muni_hist_rows = muni_hist[muni_hist["municipio"] == municipio].copy()

    # Hold slow-moving structural terms at municipio-specific long-run average
    row["income_growth"] = prev_row.get("income_growth_avg", prev_row.get("income_growth", 0.0))
    row["establishment_growth"] = prev_row.get("establishment_growth_avg", prev_row.get("establishment_growth", 0.0))
    row["lag_total_crime_rate"] = prev_row.get("lag_total_crime_rate_avg", prev_row.get("lag_total_crime_rate", 0.0))
    row["svi_pca_1"] = prev_row.get("svi_pca_1_avg", prev_row.get("svi_pca_1", 0.0))

    wind_sum, quake_sum = compute_recent_hazard_memory(muni_hist_rows, year, scenario, municipio)
    row["wind_3yr_sum"] = wind_sum
    row["seismic_3yr_sum"] = quake_sum
    row["wind_3yr_x_svi"] = row["wind_3yr_sum"] * row["svi_pca_1"]
    row["seismic_3yr_x_svi"] = row["seismic_3yr_sum"] * row["svi_pca_1"]

    return row


def recursive_forecast(model, latest_rows, scenario_name, scenario, residuals, n_sims=300):
    forecast_years = list(range(forecast_start_year, forecast_end_year + 1))
    sim_records = []
    median_records = []

    for sim in range(n_sims):
        current = latest_rows.copy()
        current["sim_id"] = sim

        path_rows = []

        for year in forecast_years:
            next_rows = []

            for _, prev_row in current.iterrows():
                future_row = build_future_row(prev_row, year, scenario, df)

                X_future = pd.DataFrame([future_row[all_features]])
                pred_change = float(model.predict(X_future)[0])

                # Add stochastic variation from historical residuals
                resid = float(np.random.choice(residuals)) if len(residuals) else 0.0
                pred_change_sim = pred_change + resid * 0.35

                prev_pop = float(prev_row["total_population"])
                next_pop = prev_pop * (1 + pred_change_sim / 100.0)
                next_pop = max(0, next_pop)

                future_row["predicted_pop_change_1y"] = pred_change_sim
                future_row["total_population"] = next_pop

                # Update lag structure recursively
                future_row["lag_pop_change_2"] = prev_row.get("lag_pop_change_1", np.nan)
                future_row["lag_pop_change_1"] = pred_change_sim

                future_row["scenario_name"] = scenario_name
                future_row["scenario_label"] = scenario["label"]
                future_row["sim_id"] = sim
                next_rows.append(future_row)
                path_rows.append(future_row)

            current = pd.DataFrame(next_rows)

        sim_df = pd.DataFrame(path_rows)
        sim_records.append(sim_df)

    all_sim_df = pd.concat(sim_records, ignore_index=True)

    # Island summary per simulation path
    island_sim = (
        all_sim_df.groupby(["scenario_name", "scenario_label", "sim_id", "year"], as_index=False)["total_population"]
        .sum()
        .rename(columns={"total_population": "island_population"})
    )

    # Region summary per simulation path
    region_sim = (
        all_sim_df.groupby(["scenario_name", "scenario_label", "sim_id", "year", "region"], as_index=False)["total_population"]
        .sum()
        .rename(columns={"total_population": "region_population"})
    )

    # Municipio summary per simulation path
    muni_sim = (
        all_sim_df.groupby(["scenario_name", "scenario_label", "sim_id", "year", "municipio"], as_index=False)["total_population"]
        .sum()
        .rename(columns={"total_population": "municipio_population"})
    )

    return all_sim_df, island_sim, region_sim, muni_sim


In [ ]:

# Run all scenarios
scenario_outputs = {}

for scen_name, scen in scenario_definitions.items():
    print("Running scenario:", scen_name)
    all_sim_df, island_sim, region_sim, muni_sim = recursive_forecast(
        model=annual_model,
        latest_rows=latest_rows,
        scenario_name=scen_name,
        scenario=scen,
        residuals=residuals,
        n_sims=n_sims
    )
    scenario_outputs[scen_name] = {
        "detail": all_sim_df,
        "island_sim": island_sim,
        "region_sim": region_sim,
        "muni_sim": muni_sim
    }

print("Finished scenarios:", list(scenario_outputs.keys()))


Running scenario: baseline_no_disaster


In [ ]:

def summarize_distribution(df_sim, value_col, group_cols):
    summary = (
        df_sim.groupby(group_cols)[value_col]
        .quantile([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
        .unstack()
        .reset_index()
    )
    summary.columns = group_cols + ["p05", "p10", "p25", "p50", "p75", "p90", "p95"]
    return summary

# Create summary outputs
all_island_summaries = []
all_region_summaries = []
all_municipal_summaries = []

for scen_name, out in scenario_outputs.items():
    island_summary = summarize_distribution(
        out["island_sim"], "island_population", ["scenario_name", "scenario_label", "year"]
    )
    region_summary = summarize_distribution(
        out["region_sim"], "region_population", ["scenario_name", "scenario_label", "region", "year"]
    )
    municipal_summary = summarize_distribution(
        out["muni_sim"], "municipio_population", ["scenario_name", "scenario_label", "municipio", "year"]
    )

    all_island_summaries.append(island_summary)
    all_region_summaries.append(region_summary)
    all_municipal_summaries.append(municipal_summary)

island_scenario_summary = pd.concat(all_island_summaries, ignore_index=True)
region_scenario_summary = pd.concat(all_region_summaries, ignore_index=True)
municipal_scenario_summary = pd.concat(all_municipal_summaries, ignore_index=True)

save_csv(island_scenario_summary, "phase3_island_scenario_summary.csv")
save_csv(region_scenario_summary, "phase3_region_scenario_summary.csv")
save_csv(municipal_scenario_summary, "phase3_municipal_scenario_summary.csv")

island_scenario_summary.head()


In [ ]:

# Impact table: compare each shock scenario against baseline at horizon year
baseline_name = "baseline_no_disaster"
baseline_2030 = (
    island_scenario_summary[
        (island_scenario_summary["scenario_name"] == baseline_name) &
        (island_scenario_summary["year"] == forecast_end_year)
    ][["p05", "p10", "p25", "p50", "p75", "p90", "p95"]]
    .iloc[0]
)

impact_rows = []
for scen_name, scen in scenario_definitions.items():
    scenario_2030 = island_scenario_summary[
        (island_scenario_summary["scenario_name"] == scen_name) &
        (island_scenario_summary["year"] == forecast_end_year)
    ]
    if len(scenario_2030) == 0:
        continue
    row = scenario_2030.iloc[0]
    impact_rows.append({
        "scenario_name": scen_name,
        "scenario_label": row["scenario_label"],
        "forecast_year": forecast_end_year,
        "median_population": row["p50"],
        "median_vs_baseline": row["p50"] - baseline_2030["p50"],
        "p05_vs_baseline": row["p05"] - baseline_2030["p05"],
        "p95_vs_baseline": row["p95"] - baseline_2030["p95"],
    })

island_impact_summary = pd.DataFrame(impact_rows).sort_values("median_population", ascending=False)
save_csv(island_impact_summary, "phase3_island_impact_summary.csv")
island_impact_summary


## Plotting helpers

In [ ]:

def plot_fan(ax, hist_df, forecast_df, scenario_title, ylabel="Population"):
    # Historical
    ax.plot(hist_df["year"], hist_df["population"], color="black", marker="o", linewidth=1.5, label="Observed")

    # Forecast fan bands
    years = forecast_df["year"].values
    ax.fill_between(years, forecast_df["p05"], forecast_df["p95"], alpha=0.15, label="5%-95%")
    ax.fill_between(years, forecast_df["p10"], forecast_df["p90"], alpha=0.20, label="10%-90%")
    ax.fill_between(years, forecast_df["p25"], forecast_df["p75"], alpha=0.28, label="25%-75%")
    ax.plot(years, forecast_df["p50"], linewidth=2.2, label="Median forecast")

    ax.axvline(latest_year, linestyle="--", linewidth=1.2)
    ax.text(latest_year - 0.3, hist_df["population"].max() * 0.995, f"Current: {latest_year}", ha="right", va="top")
    ax.set_title(scenario_title)
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)


def get_island_summary_for_plot(scenario_name):
    return island_scenario_summary[
        island_scenario_summary["scenario_name"] == scenario_name
    ].sort_values("year").reset_index(drop=True)


In [ ]:

# Side-by-side island forecast plots: baseline vs hurricane / baseline vs earthquake
baseline_plot_df = get_island_summary_for_plot("baseline_no_disaster")
hurricane_plot_df = get_island_summary_for_plot(f"hurricane_cat{hurricane_category}_{shock_year}")
earthquake_plot_df = get_island_summary_for_plot(f"earthquake_m{str(earthquake_magnitude).replace('.', '')}_{shock_year}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

plot_fan(
    axes[0],
    historical_island,
    baseline_plot_df,
    "No natural disaster"
)
plot_fan(
    axes[0],
    historical_island,
    hurricane_plot_df,
    f"Hurricane Category {hurricane_category} vs baseline context"
)

# To make the first panel a direct side-by-side comparison, overlay baseline median
axes[0].plot(baseline_plot_df["year"], baseline_plot_df["p50"], linestyle="--", linewidth=2, label="Baseline median")
axes[0].legend(loc="best", fontsize=8)

plot_fan(
    axes[1],
    historical_island,
    earthquake_plot_df,
    f"Earthquake M{earthquake_magnitude}"
)
axes[1].plot(baseline_plot_df["year"], baseline_plot_df["p50"], linestyle="--", linewidth=2, label="Baseline median")
axes[1].legend(loc="best", fontsize=8)

fig.suptitle("Puerto Rico island-level forecast distribution: baseline vs hazard scenarios", y=1.02, fontsize=13)
fig.tight_layout()

save_figure(fig, "phase3_side_by_side_island_scenarios.png")
plt.show()


In [ ]:

# Cleaner pair of side-by-side comparison panels:
# left = no disaster vs hurricane
# right = no disaster vs earthquake

def plot_compare_panel(ax, hist_df, baseline_df, scenario_df, title):
    ax.plot(hist_df["year"], hist_df["population"], color="black", marker="o", linewidth=1.5, label="Observed")

    # Baseline fan
    ax.fill_between(baseline_df["year"], baseline_df["p25"], baseline_df["p75"], alpha=0.18, label="Baseline 25%-75%")
    ax.plot(baseline_df["year"], baseline_df["p50"], linewidth=2.0, linestyle="--", label="Baseline median")

    # Scenario fan
    ax.fill_between(scenario_df["year"], scenario_df["p25"], scenario_df["p75"], alpha=0.22, label="Scenario 25%-75%")
    ax.plot(scenario_df["year"], scenario_df["p50"], linewidth=2.2, label="Scenario median")

    ax.axvline(latest_year, linestyle="--", linewidth=1.2)
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Population")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", fontsize=8)

fig2, axes2 = plt.subplots(1, 2, figsize=(15, 5))

plot_compare_panel(
    axes2[0],
    historical_island,
    baseline_plot_df,
    hurricane_plot_df,
    f"No disaster vs Hurricane Category {hurricane_category} ({shock_year})"
)

plot_compare_panel(
    axes2[1],
    historical_island,
    baseline_plot_df,
    earthquake_plot_df,
    f"No disaster vs Earthquake M{earthquake_magnitude} ({shock_year})"
)

fig2.suptitle("Scenario comparison fan charts for Puerto Rico island-level population", y=1.02, fontsize=13)
fig2.tight_layout()

save_figure(fig2, "phase3_side_by_side_baseline_vs_hazard.png")
plt.show()


In [ ]:

# Region-level comparison at horizon year
region_2030 = region_scenario_summary[region_scenario_summary["year"] == forecast_end_year].copy()
region_2030_baseline = region_2030[region_2030["scenario_name"] == "baseline_no_disaster"][["region", "p50"]].rename(columns={"p50": "baseline_p50"})
region_2030 = region_2030.merge(region_2030_baseline, on="region", how="left")
region_2030["median_vs_baseline"] = region_2030["p50"] - region_2030["baseline_p50"]
save_csv(region_2030, "phase3_region_2030_comparison.csv")
region_2030.head()


In [ ]:

# Municipal-level 2030 comparison
municipal_2030 = municipal_scenario_summary[municipal_scenario_summary["year"] == forecast_end_year].copy()
municipal_2030_baseline = municipal_2030[municipal_2030["scenario_name"] == "baseline_no_disaster"][["municipio", "p50"]].rename(columns={"p50": "baseline_p50"})
municipal_2030 = municipal_2030.merge(municipal_2030_baseline, on="municipio", how="left")
municipal_2030["median_vs_baseline"] = municipal_2030["p50"] - municipal_2030["baseline_p50"]
save_csv(municipal_2030, "phase3_municipal_2030_comparison.csv")
municipal_2030.head()


## Simple interpretation tables

These tables make it easier to answer questions like:

- “If there is no natural disaster, what is the 2030 island population distribution?”
- “If there is a Category 3 hurricane in 2026, how much lower is the median 2030 population?”
- “Which regions or municipios are most affected relative to baseline?”


In [ ]:

# Compact island interpretation table
island_interpretation = island_scenario_summary[
    island_scenario_summary["year"].isin([forecast_start_year, shock_year, forecast_end_year])
].copy()

save_csv(island_interpretation, "phase3_island_interpretation_table.csv")
island_interpretation.sort_values(["year", "scenario_name"]).head(20)


In [ ]:

# Save manifest
manifest_df = pd.DataFrame(saved_files)
save_csv(manifest_df, "phase3_saved_files_manifest.csv")
manifest_df
